# Q1: Which past indicators predict future KSI crashes?

Builds a validated KSI risk score for every crash site (issue #19). All logic lives in the layers; this notebook only wires them together.

**Design decisions** (each one is set in config):
- **Sites:** DBSCAN clusters of geocoded crash locations (eps 80 ft), built only from crashes before 2024 (`SiteConfig`).
- **Outcome:** KSI crashes per site-year. KSI means a death, or a severe-injury proxy from the persons table's `complaint`/`emotional_status`, because public data has no "seriously injured" field (`CrashConfig`).
- **Predictors:** 3-year lagged counts of KSI, fatal, injury and all crashes and of pedestrian and cyclist casualties, entered as `log1p`, plus highway share and camera flags (`PanelConfig`).
- **Validation:** expanding window. For each test year (2024, 2025), fit on earlier years and score that year. Metrics are reported for all sites and for non-highway sites (`ValidationConfig`).

In [1]:
import logging
import sys
from pathlib import Path

sys.path.append("..")

import pandas as pd

from model.count_models import HistoryRanker, NegativeBinomialModel
from preprocessing.panel import load_panel
from validation.validate import ValidationConfig, evaluate

logging.basicConfig(level=logging.INFO, format="%(message)s")
pd.set_option("display.precision", 3)

In [2]:
panel, scoring, sites = load_panel()
panel.shape, scoring.shape, sites.shape

clean_crashes: dropped 0 crashes before 2017-01-01


drop_ungeocoded: dropped 95967 of 1311313 crashes (7.3%) with missing or out-of-NYC coordinates


flag_highway: 146081 of 1215346 crashes flagged as highway


flag_ksi: 25873 KSI crashes (2107 with a death); 3533 crashes had no person records


build_site_points: 1013673 crashes before 2024-01-01 at 210488 unique points -> 80611 sites


assign_sites: 18974 crashes unassigned (18974 of 201673 on/after 2024-01-01)


count_site_years: 15 crashes with missing injury counts treated as 0


assign_cameras: 4191 of 4610 cameras matched to a site within 150 ft


load_panel: 483666 site-years for 80611 sites


((483666, 24), (80611, 24), (80611, 8))

## Model ladder
Rung 1 is current practice (rank by past KSI). If a crash-history model ranks better, past KSI alone is too sparse a signal.

In [3]:
LAGS = [c for c in panel if c.startswith("log1p_")]
CONTEXT = ["highway_share", "rlc_active", "speed_active"]

ladder = {
    "1 past KSI (baseline)": lambda: HistoryRanker("ksi_lag3", n_years=3),
    "2 past injury crashes": lambda: HistoryRanker("injury_lag3", n_years=3),
    "3 NB: KSI history": lambda: NegativeBinomialModel(["log1p_ksi_lag3"]),
    "4 NB: crash history": lambda: NegativeBinomialModel(LAGS),
    "5 NB: + highway, cameras": lambda: NegativeBinomialModel(LAGS + CONTEXT),
}
config = ValidationConfig()

results, calibrations = [], []
for name, make_model in ladder.items():
    metrics, calibration, _ = evaluate(make_model, panel, config)
    results.append(metrics.assign(model=name))
    calibrations.append(calibration.assign(model=name))

# Sensitivity: drop every site that ever had a camera
no_camera = panel[~panel["camera_site"]]
for name in ["1 past KSI (baseline)", "4 NB: crash history"]:
    metrics, _, _ = evaluate(ladder[name], no_camera, config)
    results.append(metrics.assign(model=f"no-camera sites: {name}"))

results = pd.concat(results)
calibrations = pd.concat(calibrations)

### Share of next year's KSI crashes captured by the top-k sites (mean of test years)

In [4]:
(
    results[results["metric"] == "top_k_capture"]
    .pivot_table(index=["scope", "model"], columns="k", values="value")
)

k                                                   50.0    100.0   250.0   \
scope       model                                                            
all         1 past KSI (baseline)                    0.024   0.037   0.064   
            2 past injury crashes                    0.027   0.043   0.079   
            3 NB: KSI history                        0.024   0.037   0.064   
            4 NB: crash history                      0.029   0.044   0.081   
            5 NB: + highway, cameras                 0.029   0.043   0.081   
            no-camera sites: 1 past KSI (baseline)   0.025   0.039   0.063   
            no-camera sites: 4 NB: crash history     0.031   0.048   0.082   
non_highway 1 past KSI (baseline)                    0.028   0.042   0.072   
            2 past injury crashes                    0.030   0.048   0.088   
            3 NB: KSI history                        0.028   0.042   0.072   
            4 NB: crash history                      0.033   0.049   0.089   
            5 NB: + highway, cameras                 0.033   0.049   0.090   
            no-camera sites: 1 past KSI (baseline)   0.029   0.043   0.069   
            no-camera sites: 4 NB: crash history     0.035   0.051   0.086   

k                                                   500.0   1000.0  
scope       model                                                   
all         1 past KSI (baseline)                    0.096   0.146  
            2 past injury crashes                    0.123   0.198  
            3 NB: KSI history                        0.096   0.146  
            4 NB: crash history                      0.128   0.199  
            5 NB: + highway, cameras                 0.127   0.198  
            no-camera sites: 1 past KSI (baseline)   0.091   0.143  
            no-camera sites: 4 NB: crash history     0.127   0.200  
non_highway 1 past KSI (baseline)                    0.104   0.158  
            2 past injury crashes                    0.135   0.210  
            3 NB: KSI history                        0.104   0.158  
            4 NB: crash history                      0.135   0.209  
            5 NB: + highway, cameras                 0.134   0.207  
            no-camera sites: 1 past KSI (baseline)   0.098   0.147  
            no-camera sites: 4 NB: crash history     0.135   0.206

### Fit: Poisson deviance (lower is better) and predicted / observed total KSI

In [5]:
(
    results[results["metric"] != "top_k_capture"]
    .pivot_table(index=["scope", "model"], columns="metric", values="value")
)

metric                                              poisson_deviance  \
scope       model                                                      
all         1 past KSI (baseline)                              0.584   
            2 past injury crashes                              0.904   
            3 NB: KSI history                                  0.205   
            4 NB: crash history                                0.166   
            5 NB: + highway, cameras                           0.166   
            no-camera sites: 1 past KSI (baseline)             0.543   
            no-camera sites: 4 NB: crash history               0.155   
non_highway 1 past KSI (baseline)                              0.539   
            2 past injury crashes                              0.833   
            3 NB: KSI history                                  0.192   
            4 NB: crash history                                0.156   
            5 NB: + highway, cameras                           0.155   
            no-camera sites: 1 past KSI (baseline)             0.499   
            no-camera sites: 4 NB: crash history               0.145   

metric                                              pred_to_obs_total  
scope       model                                                      
all         1 past KSI (baseline)                               1.006  
            2 past injury crashes                              13.767  
            3 NB: KSI history                                   1.041  
            4 NB: crash history                                 1.018  
            5 NB: + highway, cameras                            1.018  
            no-camera sites: 1 past KSI (baseline)              1.006  
            no-camera sites: 4 NB: crash history                1.022  
non_highway 1 past KSI (baseline)                               1.031  
            2 past injury crashes                              13.751  
            3 NB: KSI history                                   1.114  
            4 NB: crash history                                 1.044  
            5 NB: + highway, cameras                            1.052  
            no-camera sites: 1 past KSI (baseline)              1.033  
            no-camera sites: 4 NB: crash history                1.052

### Calibration of the crash-history model by decile of predicted risk (all sites)

In [6]:
calibrations.query("model == '4 NB: crash history' and scope == 'all'").pivot_table(
    index="bin", columns="test_year", values=["mean_pred", "mean_obs"]
)

mean_obs        mean_pred       
test_year     2024   2025      2024   2025
bin                                       
1            0.005  0.004     0.007  0.006
2            0.003  0.004     0.007  0.006
3            0.004  0.005     0.007  0.006
4            0.005  0.004     0.010  0.007
5            0.005  0.007     0.011  0.010
6            0.007  0.010     0.014  0.013
7            0.016  0.014     0.018  0.017
8            0.027  0.028     0.027  0.025
9            0.062  0.059     0.048  0.046
10           0.197  0.182     0.192  0.185

## Which indicators carry the signal?
Rate ratios from the crash-history model fit on every year. Standard errors are clustered by site. The lag counts are nested (KSI ⊂ injury ⊂ all crashes), so read each coefficient as the effect *given the others*.

In [7]:
final = NegativeBinomialModel(LAGS).fit(panel, panel["ksi"], groups=panel["site_id"])
final.coefficients()

,coef,rate_ratio,ci_low,ci_high,p_value
const,-5.526,0.004,0.004,0.004,0.000e+00
log1p_ksi_lag3,-0.063,0.939,0.897,0.984,8.311e-03
log1p_fatal_lag3,-0.042,0.959,0.855,1.075,4.707e-01
log1p_injury_lag3,0.445,1.561,1.491,1.633,1.124e-81
log1p_crashes_lag3,0.740,2.097,2.021,2.175,0.000e+00
log1p_ped_injured_lag3,0.063,1.065,1.031,1.099,1.130e-04
log1p_cyc_injured_lag3,0.118,1.125,1.086,1.165,3.638e-11
year_2021,0.003,1.003,0.950,1.060,9.016e-01
year_2022,0.188,1.207,1.141,1.277,4.867e-11
year_2023,0.455,1.576,1.488,1.668,9.449e-55


## Risk score for every site
Expected KSI crashes in the year after the data ends, from the model above (the latest year's fixed effect is carried forward). The file is written to `experiments/outputs/`, which git ignores.

In [8]:
risk = (
    scoring[["site_id", "year", "ksi_lag3", "injury_lag3", "crashes_lag3", "highway_share", "camera_site"]]
    .assign(expected_ksi=final.predict(scoring))
    .merge(sites[["site_id", "latitude", "longitude", "n_points", "span_ft"]], on="site_id")
    .sort_values("expected_ksi", ascending=False)
    .assign(rank=lambda d: range(1, len(d) + 1))
)
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
risk.to_parquet(out_dir / f"q1_site_risk_{risk['year'].iloc[0]}.parquet")
risk.head(25)

,site_id,year,ksi_lag3,injury_lag3,crashes_lag3,highway_share,camera_site,expected_ksi,latitude,longitude,n_points,span_ft,rank
15511,15511,2026,9.0,144.0,324.0,2.888e-02,True,6.858,40.719,-73.988,105,847.553,1
13060,13060,2026,13.0,134.0,354.0,5.615e-04,False,6.772,40.717,-73.997,335,1167.638,2
15632,15632,2026,11.0,98.0,222.0,1.010e-03,False,4.163,40.759,-73.989,85,1156.692,3
14792,14792,2026,8.0,92.0,175.0,0.000e+00,False,3.502,40.760,-73.992,113,1486.305,4
31886,31886,2026,8.0,95.0,189.0,0.000e+00,False,3.426,40.698,-73.936,70,761.727,5
32784,32784,2026,4.0,82.0,151.0,0.000e+00,False,2.832,40.797,-73.935,74,769.968,6
23625,23625,2026,4.0,80.0,136.0,0.000e+00,False,2.499,40.657,-73.960,117,1217.021,7
37195,37195,2026,12.0,80.0,145.0,0.000e+00,False,2.473,40.817,-73.921,109,873.567,8
34387,34387,2026,2.0,73.0,149.0,2.983e-01,False,2.438,40.668,-73.931,45,681.179,9
46322,46322,2026,2.0,74.0,163.0,0.000e+00,True,2.435,40.676,-73.897,31,370.490,10
